## Setup & Initialization

In [ ]:
import random
import pyspark
from delta.tables import DeltaTable, IdentityGenerator
from pyspark.sql import SparkSession, functions as F
from datetime import datetime
import ConnectionConfig as cc
from importlib import reload

from pyspark.sql.functions import *

In [ ]:
reload(cc)

print(cc.__file__)

cc.setupEnvironment()
cc.listEnvironment()

In [ ]:
spark = cc.startLocalCluster("factTreasureHunt")
spark.getActiveSession()

## Load dimension tables to get surrogate keys

In [ ]:
dim_user = spark.read.format("delta").load("spark-warehouse/dim_user")
dim_treasure = spark.read.format("delta").load("spark-warehouse/dim_treasure")
dim_date = spark.read.format("delta").load("spark-warehouse/dim_date")
dim_season = spark.read.format("delta").load("spark-warehouse/dim_season")
dim_weather = spark.read.format("delta").load("spark-warehouse/dim_weather")
dim_hour = spark.read.format("delta").load("spark-warehouse/dim_hour")

In [ ]:
dim_season.show()

In [ ]:
dim_weather.show()

# Reading the source table

In [ ]:
cc.set_connectionProfile("catchem")

print(cc.create_jdbc())

In [ ]:
df_operational_user = cc.read_postgres_without_partition(spark, "user_table")
df_operational_treasure = cc.read_postgres_without_partition(spark, "treasure")
df_operational_treasure_log = cc.read_postgres_by_date(spark, "treasure_log", "log_time")
df_operational_city = cc.read_postgres_without_partition(spark, "city")
df_operational_treasure_stages = cc.read_postgres_without_partition(spark, "treasure_stages")

print('user size: ', df_operational_user.count())  ## 486523
print('treasure size: ', df_operational_treasure.count())  ## 128294
print('treasure_log size: ', df_operational_treasure_log.count())  ## 1789438
print('city size: ', df_operational_city.count())  ## 1264588
print('treasure_stages size: ', df_operational_treasure_stages.count())  ## 504119

### Selecting and renaming columns for easier joins and transformations; also, we can filter or aggregate here if needed to optimize later steps

In [ ]:
df_operational_city = df_operational_city.select(
    "city_id",
    "postal_code",
)

df_operational_user = df_operational_user.select(
    F.col("id").alias("user_id"),
    F.col("city_city_id").alias("city_id")
)

df_operational_treasure_log = (
    df_operational_treasure_log
    .filter(F.col("log_time").isNotNull())
    .withColumn("duration_minutes",
                (F.unix_timestamp("log_time") - F.unix_timestamp("session_start")) / 60
                )
    .withColumn(
        "hour",
        F.hour(F.date_trunc("hour", F.col("log_time") + F.expr("INTERVAL 30 MINUTES")))
        #todo round to nearest hour
        # F.hour("log_time")
    )
    .withColumn("full_date", F.to_date("log_time"))
    .select(
        F.col("hunter_id").alias("user_id"),
        "duration_minutes",
        "treasure_id",
        "hour",
        "full_date"
    )
)

df_operational_treasure = df_operational_treasure.select(
    F.col("id").alias("treasure_id"),
    "difficulty",
    "terrain"
)

df_operational_treasure_stages = df_operational_treasure_stages \
    .groupBy("treasure_id") \
    .agg(F.count("stages_id").alias("stage_count"))

print('city size: ', df_operational_city.count())  ## 1264588
print('user size: ', df_operational_user.count())  ## 486523
print('treasure_log size: ', df_operational_treasure_log.count())  ## 1789438
print('treasure size: ', df_operational_treasure.count())  ## 128294

In [ ]:
## Joining the stage count back to the treasure table, filling missing stage counts with 0
df_operational_treasure = (
    df_operational_treasure
    .join(df_operational_treasure_stages, "treasure_id", "left")
    .select(
        "treasure_id",
        "difficulty",
        "terrain",
        F.coalesce(F.col("stage_count"), F.lit(0)).cast("integer").alias("stage_count")
    )
)

df_operational_treasure.show(10, truncate=False)

In [ ]:
df_operational_treasure_log.show(10, truncate=False)

# Extracting weather information the json files

In [ ]:
# Load weather JSON files
weather_json_path = "./data/weather_data/*.json"
weather_json = spark.read.option("multiLine", True).json(weather_json_path)

In [ ]:
# This returns a Python list of the column names
print(weather_json.columns)

In [ ]:
# Explode the weather array to get individual weather entries
weather_json.show(10, truncate=False)

In [ ]:
weather_json.count() #243

In [ ]:
weather_df = (
    weather_json
    .withColumn("weather_entry", F.explode("weather"))
    .withColumn("ts", F.to_timestamp(F.from_unixtime("dt")))
    .select(
        F.col("zipCode").alias("postal_code"),
        "ts",
        F.col("weather_entry.id").alias("weather_id")
    )
    .withColumn(
        "hour",
        F.hour(F.date_trunc("hour", F.col("ts") + F.expr("INTERVAL 30 MINUTES")))
        #todo round to nearest hour
        # F.hour("ts")
    )
    .withColumn(
        "full_date",
        F.to_date("ts")
    )
    .drop("ts")
)

weather_df.show(10, truncate=False)

In [ ]:
df_fact_raw = (
    df_operational_treasure_log.alias("log")
    .join(df_operational_user.alias("u"), "user_id", "left")
    .join(df_operational_city.alias("c"), "city_id", "left")
    .join(df_operational_treasure.alias("t"), "treasure_id", "left")
    .join(dim_user.alias("du").select("user_id", "user_sk"), "user_id", "left")
    .join(dim_treasure.alias("dt"), ["difficulty", "terrain", "stage_count"], "left")
    .join(dim_date.alias("dd"), "full_date", "left")
    .join(dim_hour.alias("dh"), "hour", "left")
    .join(weather_df.alias("w"), ["postal_code", "hour", "full_date"], "left")
)

df_fact_raw.show(10)

In [ ]:
df_fact_raw.filter(F.col("treasure_sk").isNull()).count()  ## 0, all treasures matched

### Join to weather dimension using the weather_id -> weather_sk mapping logic

In [ ]:
df_fact_raw = (
    df_fact_raw
    .withColumn(
        "weather_name",
        F.when((F.col("weather_id") >= 200) & (F.col("weather_id") <= 699), "Rain")
        .when(F.col("weather_id").isNotNull(), "No Rain")
        .otherwise("Weather Unknown")
    )
)

In [ ]:
df_fact_raw = (
    df_fact_raw
    .join(dim_weather.alias("dw"), df_fact_raw.weather_name == F.col("dw.name"), "left")
)

df_fact_raw = df_fact_raw.drop("weather_name")

df_fact_raw.show(10, truncate=False)

In [ ]:
df_weather_counts = (
    df_fact_raw
    .groupBy("weather_sk")
    .agg(F.count("*").alias("count"))
    .orderBy(F.col("count").desc())
)

df_weather_counts.show()

### Join to season dimension using the month -> season_sk mapping logic (based on standard meteorological seasons)

In [ ]:
df_fact_raw = (
    df_fact_raw
    .withColumn(
        "season_name",
        F.when(F.month("full_date").isin(12, 1, 2), "Winter")
        .when(F.month("full_date").isin(3, 4, 5), "Spring")
        .when(F.month("full_date").isin(6, 7, 8), "Summer")
        .otherwise("Autumn")
    )
)

In [ ]:
df_fact_raw = (
    df_fact_raw
    .join(dim_season.alias("ds"), "season_name", "left")
)

df_fact_raw.show(10, truncate=False)

In [ ]:
df_season_counts = (
    df_fact_raw
    .groupBy("season_sk")
    .agg(F.count("*").alias("count"))
    .orderBy(F.col("count").desc())
)

df_season_counts.show()

## Deleting the dimension table if it already exists

In [ ]:
cc.delete_delta_table(spark, "fact_treasure_hunt")

## Creating the fact table

In [ ]:
from pyspark.sql.types import LongType, IntegerType

# Initialize the Fact table
DeltaTable.create(spark) \
    .tableName("fact_treasure_hunt") \
    .addColumn("treasure_hunt_sk", LongType(), nullable=False, generatedAlwaysAs=IdentityGenerator(0, 1)) \
    .addColumn("user_sk", LongType(), nullable=False) \
    .addColumn("treasure_sk", IntegerType(), nullable=False) \
    .addColumn("date_SK", LongType(), nullable=False) \
    .addColumn("season_sk", IntegerType(), nullable=False) \
    .addColumn("hour_sk", IntegerType(), nullable=False) \
    .addColumn("weather_sk", LongType(), nullable=False) \
    .addColumn("duration_minutes", IntegerType(), nullable=False) \
    .property("delta.feature.identityColumns", "supported") \
    .execute()

In [ ]:
df_fact_final = (
    df_fact_raw
    .select(
        F.col("user_sk").cast(LongType()),
        F.col("treasure_sk").cast(IntegerType()),
        F.col("date_sk").cast(LongType()),
        F.col("season_sk").cast(IntegerType()),
        F.col("hour_sk").cast(IntegerType()),
        F.col("weather_sk").cast(LongType()),
        F.col("duration_minutes").cast(IntegerType())
    )
    .fillna(0)
)
df_fact_final.show(10, truncate=False)

## Populating the fact table

In [ ]:
df_fact_final.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("fact_treasure_hunt")

In [ ]:
# Delete parquet file if it already exists
cc.delete_parquet_table("parquet/fact_treasure_hunt_parquet")

In [ ]:
# Save as Parquet format

df = spark.read.format("delta").load("spark-warehouse/fact_treasure_hunt")

df.coalesce(1).write.format("parquet").save("./parquet/fact_treasure_hunt_parquet")

## Checking the results

In [ ]:
spark.read.format("delta").load("spark-warehouse/fact_treasure_hunt").show(10, truncate=False)

## Cleaning up the environment (stop Spark and reset variables)

In [ ]:
spark.stop()

In [ ]:
%reset